# Мониторинг фин. эффекта по Excel

Оценка без факта ПСР:

```
e_fee = p_fu×100k + p_court×(100k+15k)
expected_psr = precision × Σ_I (paid×k + e_fee)
net = expected_psr − Σ_I paid
```

Интервенция `I`: `РезультатПроверки=1` ∧ `Заключено соглашение=1` ∧ `Выплата по модели=1`.

`PAID_COL` по explore: **`СуммаПлатежа`** (бьётся с `СуммаКВыплате` ~90%; рекомендованная на I не совпадает).  
Приоры: `querulus_train_dataset.parquet` на контуре → пересчёт в `retro_priors.json`; иначе демо-JSON.  
Без `preds_cf` в parquet `precision` будет fallback 0.5.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
for _p in (SRC, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

NOTEBOOK_DIR = PROJECT_ROOT / "monitoring" / "fin_effects"
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT", PROJECT_ROOT)
print("DATA_DIR", DATA_DIR)

In [ ]:
from IPython.display import display

from querulus.fin_effect.excel_monitoring import (
    default_demo_priors,
    estimate_monitoring_effect,
    load_excel,
    load_retro_priors,
    save_retro_priors,
    sensitivity_table,
    write_synthetic_claims_excel,
)

# Боевой Excel на контуре; синтетика — только если файла нет и GENERATE_SYNTHETIC=True.
GENERATE_SYNTHETIC = True
EXCEL_PATH = DATA_DIR / "querulus_claims_synthetic.xlsx"
# EXCEL_PATH = Path("/home/jovyan/old_home/querulus/monitoring/fin_effects/data/claims_prod.xlsx")

PRIORS_PATH = DATA_DIR / "retro_priors.json"
# Explore: СуммаПлатежа ≈ СуммаКВыплате; рекомендованная на I не совпадает → paid = платёж.
PAID_COL = "СуммаПлатежа"

# Зрелый train-датасет Querulus (таргеты для k / долей путей). preds_cf опционален.
RETRO_PARQUET = Path(
    "/home/jovyan/old_home/querulus/data/processed/querulus_train_dataset.parquet"
)
# RETRO_PARQUET = PROJECT_ROOT / "data" / "processed" / "querulus_train_dataset.parquet"
RETRO_THRESHOLD = 0.5

if GENERATE_SYNTHETIC and not Path(EXCEL_PATH).exists():
    write_synthetic_claims_excel(EXCEL_PATH, n_rows=300)
    print("synthetic written", EXCEL_PATH)

if not PRIORS_PATH.exists():
    save_retro_priors(default_demo_priors(), PRIORS_PATH)
    print("demo priors written", PRIORS_PATH)

df = load_excel(EXCEL_PATH)
print("shape", df.shape)
print("RETRO_PARQUET exists", Path(RETRO_PARQUET).exists())

In [ ]:
import pandas as pd
from querulus.fin_effect.excel_monitoring import compute_retro_priors

if RETRO_PARQUET is not None and Path(RETRO_PARQUET).exists():
    retro = pd.read_parquet(RETRO_PARQUET)
    print("retro shape", retro.shape)
    print(
        "has preds_cf",
        "preds_cf" in retro.columns,
        "| TARGET_FREQ",
        "TARGET_FREQ" in retro.columns,
    )
    priors = compute_retro_priors(retro, threshold=RETRO_THRESHOLD)
    save_retro_priors(priors, PRIORS_PATH)
    print("priors from parquet →", PRIORS_PATH)
else:
    priors = load_retro_priors(PRIORS_PATH)
    print("priors from json (parquet missing):", RETRO_PARQUET)

print(priors)
print("e_fee", round(priors.expected_fee(), 2))
if "preds_cf" not in getattr(locals().get("retro", pd.DataFrame()), "columns", []):
    print("note: without preds_cf precision may be fallback 0.5")

In [ ]:
effect = estimate_monitoring_effect(df, priors, paid_col=PAID_COL)
summary = {
    "paid_column": effect.paid_column,
    "n_intervention": effect.n_intervention,
    "sum_paid": round(effect.sum_paid, 2),
    "e_fee": round(effect.e_fee, 2),
    "expected_psr": round(effect.expected_psr, 2),
    "cost": round(effect.cost, 2),
    "net": round(effect.net, 2),
}
display(summary)

sens = sensitivity_table(df, priors, paid_col=PAID_COL)
display(sens)